# Complete Analysis

Interactive driver for the LDPC-RL analysis pipeline. Loads an experiment's `*.json.gz` artefacts into a per-experiment DuckDB and runs a configurable subset of the analyses defined in `config/analysis_definitions.json`.

This notebook is the interactive twin of two scripts:

- `scripts/run_analysis.py` — run a single analysis from the command line
- `scripts/run_experiment_report.py` — render the full markdown report from `config/report_template.md`

Use the notebook for exploration; use the scripts for reproducible reports.

## How to use

1. Set `EXPERIMENT_ID` to a timestamp present in your results directory (e.g. one of the `*_training_epoch1_phase1.json.gz` files).
2. Edit `SELECTED_ANALYSES` — pick from `config/analysis_definitions.json`, or run `PYTHONPATH=. python scripts/run_analysis.py --list` for the full menu.
3. Toggle `REBUILD_DATABASE` if the source files changed since the last run.
4. Toggle `SAVE_RESULTS_TO_FILE` to dump tabular results to a text file under the results directory.
5. Run all cells.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists() and (_p / 'src').exists():
        sys.path.insert(0, str(_p))
        break

import json
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.analysis.analysis import LDPCAnalysis
from src.analysis.experiment_loader import open_experiment
from src.utils.path_utils import find_project_root, resolve_config_path, resolve_results_path

PROJECT_ROOT = find_project_root()
RESULTS_DIR = resolve_results_path()
DEFINITIONS_PATH = resolve_config_path('analysis_definitions.json')

## Configuration

In [ ]:
# TODO: set this to the timestamp of the experiment you want to analyse.
# Browse the results directory or run scripts/run_analysis.py to see available IDs.
EXPERIMENT_ID = ''

REBUILD_DATABASE = False     # rebuild the .db from json.gz files
SAVE_RESULTS_TO_FILE = False # also write tabular results to a .txt next to the .db

# Pick from config/analysis_definitions.json — run scripts/run_analysis.py --list for the full menu.
SELECTED_ANALYSES = [
    'overall_performance_summary',
    'curriculum_phase_total_steps',
    'training_error_distribution',
    'evaluation_error_distribution',
    'steps_distribution_training',
    'steps_distribution',
    'success_rate_by_phase',
    'rl_success_baseline_failure_cases',
]

if not EXPERIMENT_ID:
    raise ValueError('Set EXPERIMENT_ID above before running the rest of the notebook.')

print(f'Experiment ID     : {EXPERIMENT_ID}')
print(f'Results directory : {RESULTS_DIR}')
print(f'Selected analyses : {len(SELECTED_ANALYSES)}')

## Database initialisation

Loading is delegated to `src.analysis.experiment_loader.open_experiment`, the same
helper `scripts/run_analysis.py` uses, so the notebook and the command line agree on
what counts as a readable run. It reads (epoch, phase) training segments, their
flush files, the single training file a run without progressive saving writes, and
one evaluation file per agent; loads a run that produced only one of the two halves;
and refuses a database belonging to a different experiment.

In [ ]:
# A run with `run_training: false` scores an existing checkpoint and writes no
# training files; one with `run_evaluation: false` writes no evaluation files.
# Either half is enough to load, and the analyses over the missing half return
# no rows.
analyzer = open_experiment(
    DEFINITIONS_PATH,
    RESULTS_DIR / f'{EXPERIMENT_ID}.db',
    EXPERIMENT_ID,
    RESULTS_DIR,
    rebuild=REBUILD_DATABASE,
)

## Experiment configuration

The full configuration is stored as JSON inside the `experiments` table; the keys below mirror the schema of `config/experiments_config/*/experiment_config.yml`.

In [ ]:
def fetch_experiment_config(analyzer: LDPCAnalysis, experiment_id: str) -> dict:
    row = analyzer.con.execute(
        'SELECT experiment_name, configuration, created_at FROM experiments WHERE experiment_id = ?',
        [experiment_id],
    ).fetchone()
    if not row:
        print(f'No configuration row found for experiment {experiment_id}')
        return {}
    name, config_json, created_at = row
    config = json.loads(config_json) if config_json else {}
    print(f'Experiment name : {name}')
    print(f'Created at      : {created_at}')
    return config


def summarise_config(config: dict) -> None:
    if not config:
        return

    print(f"\n[top]   code_id = {config.get('code_id')}, seed = {config.get('seed')}, device = {config.get('device')}")
    print(f"        run_dataset_creation = {config.get('run_dataset_creation')}, "
          f"run_training = {config.get('run_training')}, run_evaluation = {config.get('run_evaluation')}")

    env = config.get('environment', {})
    print(f"\n[env]   max_steps = {env.get('max_steps')}, max_flips_vector = {env.get('max_flips_vector')}")

    train = config.get('training', {})
    print('\n[train]')
    for key in ('num_epochs', 'replay_buffer_size', 'batch_size', 'target_update_freq',
                'learning_rate', 'weight_decay', 'update_frequency',
                'epsilon_decay_type', 'epsilon_start', 'epsilon_min', 'epsilon_decay',
                'epsilon_phase_reset'):
        if key in train:
            print(f'        {key:<20} = {train[key]}')

    net = train.get('network', {})
    if net:
        print('\n[network]')
        for key, value in net.items():
            print(f'        {key:<20} = {value}')

    expert = config.get('expert_experience', {})
    if expert:
        print('\n[expert]')
        for key in ('use_pregenerated_file', 'pregenerated_file'):
            if key in expert:
                print(f'        {key:<20} = {expert[key]}')

    baselines = config.get('evaluation', {}).get('baseline_agents', [])
    enabled = [b['name'] for b in baselines if b.get('enabled')]
    if baselines:
        print(f'\n[eval]  baselines_enabled = {enabled}')


experiment_config = fetch_experiment_config(analyzer, EXPERIMENT_ID)
summarise_config(experiment_config)

## Run selected analyses

In [ ]:
def analysis_metadata(analyzer: LDPCAnalysis, analysis_id: str) -> dict | None:
    defs = analyzer.definitions
    if isinstance(defs, list):
        return next((a for a in defs if a.get('id') == analysis_id), None)
    return None


def run_one(analyzer: LDPCAnalysis, analysis_id: str, data_only: bool):
    meta = analysis_metadata(analyzer, analysis_id)
    title = meta.get('title', analysis_id) if meta else analysis_id
    description = meta.get('description', '') if meta else ''

    print('\n' + '-' * 80)
    print(title)
    if description:
        print(description)
    print('-' * 80)

    try:
        result = analyzer.run_analysis(analysis_id, data_only=data_only)
    except Exception as e:
        print(f'FAILED: {e}')
        return False, None

    if isinstance(result, pd.DataFrame):
        display(result)
        return True, {'analysis_id': analysis_id, 'title': title, 'type': 'dataframe', 'dataframe': result}
    if hasattr(result, 'show'):
        result.show()
        plt.show()
        return True, {'analysis_id': analysis_id, 'title': title, 'type': 'plot'}
    print(result)
    return True, {'analysis_id': analysis_id, 'title': title, 'type': 'other', 'data': str(result)}


def export_text_report(results: list[dict], experiment_id: str) -> Path:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = RESULTS_DIR / f'{experiment_id}_analysis_results_{timestamp}.txt'
    with path.open('w') as f:
        f.write(f'Experiment {experiment_id} — analysis results ({timestamp})\n')
        f.write('=' * 80 + '\n\n')
        for i, item in enumerate(results, 1):
            f.write(f"[{i}/{len(results)}] {item['title']}  ({item['analysis_id']})\n")
            f.write('-' * 80 + '\n')
            if item['type'] == 'dataframe':
                f.write(item['dataframe'].to_string(index=False) + '\n')
            elif item['type'] == 'plot':
                f.write('(plot — not exported)\n')
            else:
                f.write(str(item.get('data', '')) + '\n')
            f.write('\n')
    return path


successful, failed, captured = [], [], []
for i, analysis_id in enumerate(SELECTED_ANALYSES, 1):
    print(f'\n[{i}/{len(SELECTED_ANALYSES)}] {analysis_id}')
    ok, record = run_one(analyzer, analysis_id, data_only=SAVE_RESULTS_TO_FILE)
    (successful if ok else failed).append(analysis_id)
    if ok and record is not None:
        captured.append(record)

print('\n' + '=' * 80)
print(f'Successful: {len(successful)}/{len(SELECTED_ANALYSES)}')
if failed:
    print(f'Failed    : {failed}')

if SAVE_RESULTS_TO_FILE and captured:
    out_path = export_text_report(captured, EXPERIMENT_ID)
    print(f'Wrote {out_path}')

## Custom SQL

The analyzer exposes a thin pass-through; queries run against the same DuckDB tables consumed by the analyses (`experiments`, `episodes`, `steps`).

In [ ]:
analyzer.query("""
    SELECT phase, COUNT(*) AS episodes
    FROM   episodes
    WHERE  episode_type = 'train'
    GROUP  BY phase
    ORDER  BY phase
""")